# 2025 홀드아웃 CSV -> Parquet 변환 (RAM-safe)

`2단계_2025홀드아웃_1단계포함.csv.gz`(96컬럼, 161만행)를 청크로 읽고, 다운캐스트는 전체
합친 뒤 한 번만 수행합니다 (학습데이터 parquet 변환 때 썼던 것과 동일한 안전한 방식 -
청크별로 다운캐스트하면 같은 컬럼이 청크마다 다른 dtype으로 캐스팅되는 버그가 생겨서
이번엔 처음부터 안전하게 감).

161만행 x 96컬럼이라 학습데이터(604만행)보다 훨씬 작아서, 청크 없이도 사실 돌아갈 수
있지만 안전하게 청크로 읽습니다.

In [1]:
import os
import gc
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/kb_challenge'
HOLDOUT_INPUT = os.path.join(DRIVE_DIR, '2단계_2025홀드아웃_1단계포함.csv.gz')
HOLDOUT_OUTPUT_PARQUET = os.path.join(DRIVE_DIR, '2단계_2025홀드아웃_1단계포함.parquet')

ID_COL = '상가업소번호'
QUARTER_COL = '기준_년분기'
CHUNK_SIZE = 300_000


## 다운캐스트 함수 (int64 범위 초과 값은 강제로 int 변환 안 하게 방어)

In [2]:
INT64_MIN, INT64_MAX = np.iinfo(np.int64).min, np.iinfo(np.int64).max

def downcast_numeric_inplace(df, verbose_name=''):
    for col in df.select_dtypes(include=['float64']).columns:
        s = df[col]
        non_null = s.dropna()
        if len(non_null) == 0:
            continue

        out_of_range = (non_null < INT64_MIN) | (non_null > INT64_MAX)
        if out_of_range.any():
            print(f'  ⚠️ [{verbose_name}] \'{col}\'에 int64 범위 초과 값 {out_of_range.sum()}건 -> float 유지')

        can_be_int = (
            not s.isna().any()
            and (non_null % 1 == 0).all()
            and not out_of_range.any()
        )
        if can_be_int:
            df[col] = pd.to_numeric(s, downcast='integer')
        else:
            df[col] = pd.to_numeric(s, downcast='float')

    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')

    return df


## 청크 단위 로드 -> 합친 뒤 한 번만 다운캐스트 -> parquet 저장

In [3]:
dtype_map = {ID_COL: str, QUARTER_COL: str}
chunks = []
n_total = 0

reader = pd.read_csv(HOLDOUT_INPUT, chunksize=CHUNK_SIZE, dtype=dtype_map, low_memory=False)
for chunk in reader:
    n_total += len(chunk)
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
del chunks
gc.collect()
print(f'로드 완료: {n_total:,}행')

df = downcast_numeric_inplace(df, verbose_name='2025홀드아웃')

# object 컬럼 중 카디널리티 낮은 것만 category로 (ID는 카디널리티 높아서 제외)
for col in df.select_dtypes(include=['object']).columns:
    if col == ID_COL:
        continue
    if df[col].nunique(dropna=True) / len(df) < 0.5:
        df[col] = df[col].astype('category')

mem_mb = df.memory_usage(deep=True).sum() / 1e6
print(f'메모리 사용량: {mem_mb:.1f} MB')
print('dtype 분포:')
print(df.dtypes.value_counts())

df.to_parquet(HOLDOUT_OUTPUT_PARQUET, index=False, engine='pyarrow', compression='snappy')
print(f'저장 완료: {HOLDOUT_OUTPUT_PARQUET}')

del df
gc.collect()


로드 완료: 1,612,891행


메모리 사용량: 660.9 MB
dtype 분포:
float32     26
int8        20
int16        5
int32        3
float64      3
int64        2
category     2
category     1
object       1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
bool         1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64
저장 완료: /content/drive/MyDrive/kb_challenge/2단계_2025홀드아웃_1단계포함.parquet


0

## 다음 단계

`2025홀드아웃_통합_피처엔지니어링.ipynb`의 `HOLDOUT_PATH`를 이 parquet 파일로 바꾸면 됩니다:
```python
HOLDOUT_PATH = os.path.join(DRIVE_DIR, '2단계_2025홀드아웃_1단계포함.parquet')
...
holdout_df = pd.read_parquet(HOLDOUT_PATH)   # read_csv 대신 read_parquet
```
